# 🇬🇧 FastSpeech 2 — English TTS Training Notebook

This notebook trains a **FastSpeech 2** Text-to-Speech model on the **Svarah English** dataset.
It covers the full pipeline:
1. Installing all dependencies (MFA, phonemizer, etc.)
2. Downloading and preparing the Svarah dataset (~6 hours of English speech)
3. Running Montreal Forced Aligner (MFA) for duration alignment
4. Preprocessing mel-spectrograms, pitch, and energy features
5. Training FastSpeech 2 with variance adaptor
6. Running inference and generating variance adaptor plots
7. Saving all results to Google Drive

> **Runtime:** Select `Runtime > Change runtime type > T4 GPU` before starting.

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU!'}")
print(f"CUDA available: {torch.cuda.is_available()}")
!nvidia-smi

## Step 1 — Install Dependencies

Install all system-level and Python-level packages required for:
- **espeak-ng**: Phoneme backend for the phonemizer library
- **sox / ffmpeg**: Audio format conversion utilities
- **phonemizer**: Text-to-phoneme conversion
- **pyworld**: Pitch (F0) extraction using WORLD vocoder
- **librosa**: Audio loading and feature extraction
- **MFA (Montreal Forced Aligner)**: Forced alignment to extract phoneme durations
- **datasets / huggingface_hub**: For downloading the Svarah dataset

In [ ]:
# Install system deps
!apt-get install -y espeak-ng espeak-ng-data libespeak-ng-dev
!apt-get install -y sox ffmpeg

# Install Python packages
!pip install -q phonemizer epitran num2words tgt pyworld librosa soundfile pyyaml tqdm indic-transliteration datasets huggingface_hub seaborn

# Install MFA via conda
!conda install -c conda-forge montreal-forced-aligner -y -q
!mfa model download acoustic english_us_arpa
!mfa model download dictionary english_us_arpa

print("✅ All dependencies installed!")

In [ ]:
import os
# Mount Google Drive to save checkpoints
from google.colab import drive
drive.mount('/content/drive')

# Clone the project repo (or upload files)
# Option A: If you've pushed to GitHub already
# !git clone https://github.com/YOUR_USERNAME/FastSpeech2_Assignment.git /content/FastSpeech2

# Option B: Upload zip and extract
# from google.colab import files
# uploaded = files.upload()

# For now, set up directory structure
!mkdir -p /content/FastSpeech2/{English,Telugu,model,utils,Results/English/{test_generated_audios,variance_adaptor_plots/{duration,pitch,energy}},Results/Telugu}
os.chdir('/content/FastSpeech2')
print("✅ Working directory:", os.getcwd())

## Step 2 — Upload Project Files

Upload your project source files to Colab using **one** of the methods below:

### Option A — Upload a ZIP file
1. Zip your local `FastSpeech2_Assignment` folder: `FastSpeech2_Assignment.zip`
2. In Colab's file panel (left sidebar), click the **Upload** icon and select the zip
3. Run the next cell — it will auto-extract and copy files into `/content/FastSpeech2/`

### Option B — Clone from GitHub
If you have pushed your code to GitHub, uncomment and run:
```python
!git clone https://github.com/YOUR_USERNAME/FastSpeech2_Assignment /content/FastSpeech2
```

### Required file structure after upload:
```
/content/FastSpeech2/
├── English/
│   ├── config.yaml
│   ├── preprocess.py
│   ├── train.py
│   └── inference.py
├── model/
│   ├── fastspeech2.py
│   └── variance_adaptor.py
└── utils/
    └── audio.py
```

In [ ]:
# After uploading your zip or cloning repo, copy files to correct locations
# This cell assumes you've uploaded FastSpeech2_Assignment.zip to Colab

import zipfile, shutil

# If uploading manually:
# from google.colab import files
# uploaded = files.upload()  # Upload your zip

# Extract if zip present
if os.path.exists('/content/FastSpeech2_Assignment.zip'):
    with zipfile.ZipFile('/content/FastSpeech2_Assignment.zip', 'r') as z:
        z.extractall('/content/')
    # Copy files to working directory
    for item in os.listdir('/content/FastSpeech2_Assignment'):
        s = os.path.join('/content/FastSpeech2_Assignment', item)
        d = os.path.join('/content/FastSpeech2', item)
        if os.path.isdir(s):
            shutil.copytree(s, d, dirs_exist_ok=True)
        else:
            shutil.copy2(s, d)
    print("✅ Files extracted and copied!")
else:
    print("⚠️ Please upload FastSpeech2_Assignment.zip or clone from GitHub")
    print("   After cloning: !git clone https://github.com/USERNAME/REPO /content/FastSpeech2")

!ls /content/FastSpeech2

## Step 3 — Download Svarah Dataset

We use the [**Svarah**](https://huggingface.co/datasets/ai4bharat/Svarah) dataset from AI4Bharat,
which contains high-quality English speech recordings from Indian speakers.

- **Target:** ~6 hours total (5 hours train + 1 hour validation)
- **Sample rate:** Resampled to **22,050 Hz**
- Files are saved as individual `.wav` files with a `metadata.csv` manifest

> ⏱️ Download time: approximately **10–20 minutes** depending on Colab's network speed.

In [ ]:
import os
os.chdir('/content/FastSpeech2')

# Download Svarah dataset from HuggingFace
from datasets import load_dataset
import soundfile as sf
import numpy as np
from tqdm import tqdm

print("Downloading Svarah dataset...")
dataset = load_dataset("ai4bharat/Svarah", split="train", trust_remote_code=True)
print(f"Total samples: {len(dataset)}")
print(f"Features: {dataset.features}")

# Calculate durations and select ~6hrs total (5hr train + 1hr val)
os.makedirs('/content/FastSpeech2/data/raw/english/wavs', exist_ok=True)

TARGET_TRAIN_HOURS = 5.0
TARGET_VAL_HOURS = 1.0
TARGET_TOTAL = (TARGET_TRAIN_HOURS + TARGET_VAL_HOURS) * 3600  # seconds

accumulated = 0
selected_indices = []

for i, sample in enumerate(dataset):
    audio = sample['audio']
    duration = len(audio['array']) / audio['sampling_rate']
    if accumulated < TARGET_TOTAL:
        selected_indices.append((i, duration))
        accumulated += duration
    if accumulated >= TARGET_TOTAL:
        break

print(f"Selected {len(selected_indices)} samples = {accumulated/3600:.2f} hours")

# Split: first ~5hrs = train, last ~1hr = val
train_cutoff = TARGET_TRAIN_HOURS * 3600
train_indices, val_indices = [], []
cum = 0
for idx, dur in selected_indices:
    if cum < train_cutoff:
        train_indices.append(idx)
    else:
        val_indices.append(idx)
    cum += dur

print(f"Train: {len(train_indices)} samples | Val: {len(val_indices)} samples")

In [ ]:
import csv, librosa

metadata = []
print("Saving audio files...")

for split, indices in [("train", train_indices), ("val", val_indices)]:
    for idx in tqdm(indices, desc=f"Saving {split}"):
        sample = dataset[idx]
        audio_array = np.array(sample['audio']['array'], dtype=np.float32)
        orig_sr = sample['audio']['sampling_rate']
        text = sample['text'] if 'text' in sample else sample.get('sentence', '')
        
        # Resample to 22050
        if orig_sr != 22050:
            audio_array = librosa.resample(audio_array, orig_sr=orig_sr, target_sr=22050)
        
        file_id = f"svarah_{idx:06d}"
        wav_path = f"/content/FastSpeech2/data/raw/english/wavs/{file_id}.wav"
        sf.write(wav_path, audio_array, 22050)
        
        duration = len(audio_array) / 22050
        metadata.append({'file_id': file_id, 'text': text, 'duration': duration, 'split': split})

# Save metadata
with open('/content/FastSpeech2/data/raw/english/metadata.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['file_id', 'text', 'duration', 'split'])
    writer.writeheader()
    writer.writerows(metadata)

print(f"✅ Saved {len(metadata)} audio files and metadata.csv")

## Step 4 — Preprocessing

Preprocessing involves three stages:

1. **MFA corpus preparation** — Organise audio + transcript files into the format MFA expects
2. **MFA forced alignment** — Align phonemes to audio frames to extract **duration** labels
3. **Feature extraction** — Extract mel-spectrograms, **pitch (F0)**, and **energy** for every utterance

> ⏱️ MFA alignment typically takes **20–40 minutes** for 6 hours of audio on a Colab instance.

In [ ]:
os.chdir('/content/FastSpeech2')

# Prepare MFA corpus format: each speaker dir with wav + lab files
import shutil

mfa_corpus = '/content/FastSpeech2/data/mfa_corpus/english'
mfa_output = '/content/FastSpeech2/data/mfa_aligned/english'
os.makedirs(mfa_corpus, exist_ok=True)
os.makedirs(mfa_output, exist_ok=True)

import pandas as pd
meta = pd.read_csv('/content/FastSpeech2/data/raw/english/metadata.csv')

for _, row in tqdm(meta.iterrows(), total=len(meta), desc="Preparing MFA corpus"):
    fid = row['file_id']
    text = str(row['text']).strip()
    
    # Copy wav
    src = f"/content/FastSpeech2/data/raw/english/wavs/{fid}.wav"
    dst_wav = f"{mfa_corpus}/{fid}.wav"
    shutil.copy2(src, dst_wav)
    
    # Write transcript
    with open(f"{mfa_corpus}/{fid}.lab", 'w', encoding='utf-8') as f:
        f.write(text.lower())

print(f"✅ MFA corpus prepared: {len(meta)} files")

In [ ]:
# Run Montreal Forced Aligner
print("Running MFA alignment (this may take 20-40 minutes)...")
!mfa align /content/FastSpeech2/data/mfa_corpus/english english_us_arpa english_us_arpa /content/FastSpeech2/data/mfa_aligned/english --clean --num_jobs 4
print("✅ MFA alignment complete!")

In [ ]:
os.chdir('/content/FastSpeech2')
import sys
sys.path.insert(0, '/content/FastSpeech2')
!python English/preprocess.py --config English/config.yaml
print("✅ Preprocessing complete!")

## Step 5 — Training

Train the FastSpeech 2 model with the variance adaptor (duration, pitch, energy predictors).

**Expected training time on Colab T4:** ~8–12 hours for 200k steps

Training logs are saved to Google Drive at:
`/content/drive/MyDrive/FastSpeech2/english_training.log`

Checkpoints are saved every 10,000 steps (configured in `English/config.yaml`).

> 💡 **Tip:** If your Colab session disconnects, you can resume from the latest checkpoint by setting `restore_step` in `config.yaml`.

In [ ]:
os.chdir('/content/FastSpeech2')

# Start training (this will run for several hours on Colab T4)
# Checkpoints saved every 10000 steps to Google Drive
!python English/train.py --config English/config.yaml 2>&1 | tee /content/drive/MyDrive/FastSpeech2/english_training.log

print("✅ Training complete!")

## Step 6 — Inference and Plot Generation

After training, we:
1. Download the **HiFi-GAN** neural vocoder (converts mel-spectrograms → waveforms)
2. Run **inference** on the 5 test sentences (E1, E2a, E2b, E3, E4)
3. Generate **variance adaptor plots** (duration, pitch, energy) for each sentence
4. Display the audio and plots inline in the notebook

### Test Sentences
| ID | Sentence |
|----|----------|
| E1 | "The quick brown fox jumps over the lazy dog." |
| E2a | "She sells seashells by the seashore." |
| E2b | "How much wood would a woodchuck chuck?" |
| E3 | "To be or not to be, that is the question." |
| E4 | "Speech synthesis has come a long way with deep learning." |

In [ ]:
os.chdir('/content/FastSpeech2')

# Download HiFi-GAN vocoder
!git clone https://github.com/jik876/hifi-gan /content/hifi-gan
os.chdir('/content/hifi-gan')
!pip install -q -r requirements.txt

# Download pretrained HiFi-GAN checkpoint (LJ_V2)
!wget -q "https://drive.google.com/uc?id=1qpgI41wNXFcH-iKq1Y42JlBC9j0je8PW" -O /content/hifi-gan/generator_v2
!wget -q "https://raw.githubusercontent.com/jik876/hifi-gan/master/config_v2.json" -O /content/hifi-gan/config.json

os.chdir('/content/FastSpeech2')

# Run inference
!python English/inference.py \
  --checkpoint English/checkpoints/best_checkpoint.pth \
  --config English/config.yaml \
  --output_dir Results/English \
  --vocoder_path /content/hifi-gan

print("✅ English inference complete! Audio files and plots saved.")

In [ ]:
import IPython.display as ipd
import glob, matplotlib.pyplot as plt
from PIL import Image
import os

# Show generated audio
print("=== Generated Audio Samples ===")
for wav_file in sorted(glob.glob('/content/FastSpeech2/Results/English/test_generated_audios/*.wav')):
    print(f"\n{os.path.basename(wav_file)}")
    display(ipd.Audio(wav_file))

# Show variance adaptor plots
print("\n=== Variance Adaptor Plots ===")
fig, axes = plt.subplots(3, 5, figsize=(20, 10))
for col, sent_id in enumerate(['E1', 'E2a', 'E2b', 'E3', 'E4']):
    for row, ptype in enumerate(['duration', 'pitch', 'energy']):
        path = f'/content/FastSpeech2/Results/English/variance_adaptor_plots/{ptype}/{sent_id}_{ptype}.png'
        if os.path.exists(path):
            img = Image.open(path)
            axes[row][col].imshow(img)
            axes[row][col].axis('off')
            if row == 0:
                axes[row][col].set_title(sent_id, fontsize=12)
            if col == 0:
                axes[row][col].set_ylabel(ptype, fontsize=10)
plt.tight_layout()
plt.savefig('/content/FastSpeech2/Results/English/all_variance_plots.png', dpi=100)
plt.show()
print("✅ All results displayed!")

In [ ]:
import shutil, os

# Copy all results to Drive for safekeeping
drive_output = '/content/drive/MyDrive/FastSpeech2/Results/English'
os.makedirs(drive_output, exist_ok=True)
shutil.copytree('/content/FastSpeech2/Results/English', drive_output, dirs_exist_ok=True)

# Copy checkpoints
drive_ckpt = '/content/drive/MyDrive/FastSpeech2/English/checkpoints'
os.makedirs(drive_ckpt, exist_ok=True)
shutil.copytree('/content/FastSpeech2/English/checkpoints', drive_ckpt, dirs_exist_ok=True)

print("✅ All results and checkpoints saved to Google Drive!")
print(f"Check: /content/drive/MyDrive/FastSpeech2/")